# Home Credit Default Risk - Kaggle Classification Challenge
**MO443 - Introdução à Inteligência Artificial**  
**TP4 - Desafio Kaggle de Classificação**

This notebook implements a complete solution for the [Home Credit Default Risk](https://kaggle.com/competitions/home-credit-default-risk) competition.

### Approaches:
1. Logistic Regression (baseline)
2. XGBoost
3. LightGBM
4. CatBoost
5. Random Forest
6. Deep Learning Neural Network (PyTorch)
7. Weighted Ensemble

### Current Run Highlights:
- Feature engineering from the relational history tables plus domain ratios from the main application table
- Memory-aware preprocessing: numeric downcasting, median imputation, variance filtering, and robust scaling where needed
- 5-fold stratified out-of-fold validation for every model
- Random Forest added as a bagging model to increase diversity against the gradient boosters
- Weighted ensemble uses LightGBM (30%), XGBoost (28%), CatBoost (27%), Random Forest (10%), and Deep NN (5%)
- Latest artifacts: best single-model AUC is XGBoost at 0.7876; ensemble AUC is 0.7865 and has the best AP at 0.2832
- SHAP, permutation/importance comparison, threshold analysis, decile analysis, and submission files are written under `outputs/`


## 1. Setup & Configuration

In [ ]:
import gc
import logging
import os
import pickle
import time
import warnings

os.environ["PYTHONWARNINGS"] = "ignore"
warnings.filterwarnings("ignore")
warnings.simplefilter("ignore")
logging.getLogger("lightgbm").setLevel(logging.ERROR)
logging.getLogger("catboost").setLevel(logging.ERROR)
logging.getLogger("xgboost").setLevel(logging.ERROR)

import lightgbm as lgb
import matplotlib

matplotlib.use("Agg")

import numpy as np
import pandas as pd
import shap
import xgboost as xgb
from catboost import CatBoostClassifier
from sklearn.feature_selection import VarianceThreshold
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import confusion_matrix, roc_auc_score
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import LabelEncoder, RobustScaler

np.seterr(all="ignore")

import matplotlib.pyplot as plt
import seaborn as sns

RANDOM_STATE = 42
N_FOLDS = 5
OUT_DIR = "outputs"
FIG_DIR = OUT_DIR + "/figures"
SUB_DIR = OUT_DIR + "/submissions"
ART_DIR = OUT_DIR + "/artifacts"

for _d in (FIG_DIR, SUB_DIR, ART_DIR):
    os.makedirs(_d, exist_ok=True)


## 2. Data Loading

The notebook first uses the local `data/` directory so repeated runs do not re-download the Kaggle archive. If the files are absent, it falls back to `kagglehub`, extracts the competition archive, and reads all CSVs through a helper that downcasts numeric columns to reduce memory use.


In [ ]:
import os

LOCAL_DIR = "data"

if os.path.exists(f"{LOCAL_DIR}/application_train.csv"):
    DATA_DIR = LOCAL_DIR
else:

    import kagglehub, zipfile

    cache_path = kagglehub.competition_download("home-credit-default-risk")
    archive = os.path.join(cache_path, "home-credit-default-risk.archive")
    if os.path.exists(archive):
        with zipfile.ZipFile(archive, "r") as zf:
            zf.extractall(cache_path)
    DATA_DIR = cache_path


def _downcast(df):
    """Shrink numerics in place: float64->float32, int64->smallest int. Halves RAM."""
    fcols = df.select_dtypes("float64").columns
    if len(fcols):
        df[fcols] = df[fcols].astype("float32")
    for c in df.select_dtypes("int64").columns:
        df[c] = pd.to_numeric(df[c], downcast="integer")

    return df


def read_csv_safe(path, **kw):
    """Read CSV with encoding fallback + immediate numeric downcast."""
    try:
        df = pd.read_csv(path, **kw)
    except UnicodeDecodeError:
        df = pd.read_csv(path, encoding="latin-1", **kw)

    return _downcast(df)


## 3. Load Main Tables

In [ ]:
train = read_csv_safe(f"{DATA_DIR}/application_train.csv")
test = read_csv_safe(f"{DATA_DIR}/application_test.csv")
sub = read_csv_safe(f"{DATA_DIR}/sample_submission.csv")


## 4. Exploratory Data Analysis

In [ ]:
missing = train.isnull().sum().sort_values(ascending=False)
missing_pct = (missing / len(train) * 100).head(20)

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(
    ["Repays (0)", "Defaults (1)"],
    train["TARGET"].value_counts().values,
    color=["#2ecc71", "#e74c3c"],
)
ax.set_title("Target Distribution")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR + "/eda_target_distribution.png", dpi=120)
plt.show()
plt.close()

if "DAYS_BIRTH" in train.columns:
    train["AGE_YEARS"] = -train["DAYS_BIRTH"] / 365.25

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        train[train["TARGET"] == 0]["AGE_YEARS"],
        bins=50,
        alpha=0.6,
        label="Repays",
        color="#2ecc71",
    )
    ax.hist(
        train[train["TARGET"] == 1]["AGE_YEARS"],
        bins=50,
        alpha=0.6,
        label="Defaults",
        color="#e74c3c",
    )
    ax.set_title("Age Distribution by Target")
    ax.set_xlabel("Age (years)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR + "/eda_age_distribution.png", dpi=120)
    plt.show()
    plt.close()

if "AMT_CREDIT" in train.columns:

    fig, ax = plt.subplots(figsize=(8, 5))
    ax.hist(
        np.log1p(train[train["TARGET"] == 0]["AMT_CREDIT"]),
        bins=50,
        alpha=0.6,
        label="Repays",
        color="#2ecc71",
    )
    ax.hist(
        np.log1p(train[train["TARGET"] == 1]["AMT_CREDIT"]),
        bins=50,
        alpha=0.6,
        label="Defaults",
        color="#e74c3c",
    )
    ax.set_title("Log Credit Amount by Target")
    ax.set_xlabel("log(1 + AMT_CREDIT)")
    ax.set_ylabel("Count")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR + "/eda_credit_amount_distribution.png", dpi=120)
    plt.show()
    plt.close()

if "AGE_YEARS" in train.columns:
    del train["AGE_YEARS"]


## 5. Feature Engineering from Supplementary Tables

The historical tables are collapsed to one row per applicant (`SK_ID_CURR`). Bureau records, bureau monthly status, previous applications, POS cash balances, credit card balances, and installment payments are summarized with count/mean/min/max/sum statistics plus domain ratios such as approval/refusal rate, credit-card utilization, payment ratio, late-payment days, and payment differences.


In [ ]:
def aggregate_supplementary_tables():
    """Aggregate supplementary tables into client-level features (memory-lean)."""

    b_aggs = {
        "AMT_CREDIT_SUM": ["mean", "sum", "min", "max"],
        "AMT_CREDIT_SUM_DEBT": ["mean", "sum", "max"],
        "AMT_CREDIT_SUM_OVERDUE": ["mean", "max"],
        "CNT_CREDIT_PROLONG": ["sum"],
        "AMT_ANNUITY": ["mean", "max"],
        "DAYS_CREDIT": ["mean", "min", "max"],
        "CREDIT_DAY_OVERDUE": ["mean", "max"],
        "DAYS_CREDIT_ENDDATE": ["mean", "min", "max"],
        "DAYS_ENDDATE_FACT": ["mean", "min"],
        "AMT_CREDIT_MAX_OVERDUE": ["mean", "max"],
        "AMT_CREDIT_SUM_LIMIT": ["mean", "sum"],
        "DAYS_CREDIT_UPDATE": ["mean", "max"],
    }
    bureau = read_csv_safe(
        f"{DATA_DIR}/bureau.csv",
        usecols=["SK_ID_CURR", "CREDIT_ACTIVE", "CREDIT_TYPE"] + list(b_aggs),
    )
    g = bureau.groupby("SK_ID_CURR")
    b_agg = g.agg(b_aggs)
    b_agg.columns = ["BUR_" + "_".join(c) for c in b_agg.columns]
    b_agg["BUR_COUNT"] = g.size()
    for col, prefix in [("CREDIT_ACTIVE", "BUR_ACT"), ("CREDIT_TYPE", "BUR_TYP")]:
        d = pd.get_dummies(bureau[col], prefix=prefix)
        d["SK_ID_CURR"] = bureau["SK_ID_CURR"].values
        b_agg = b_agg.join(d.groupby("SK_ID_CURR").mean(), how="left")
    del bureau, g
    gc.collect()

    bb = read_csv_safe(
        f"{DATA_DIR}/bureau_balance.csv",
        usecols=["SK_ID_BUREAU", "MONTHS_BALANCE", "STATUS"],
    )
    bb_agg = bb.groupby("SK_ID_BUREAU").agg({"MONTHS_BALANCE": ["size", "min", "max"]})
    bb_agg.columns = ["BB_" + "_".join(c) for c in bb_agg.columns]
    d = pd.get_dummies(bb["STATUS"], prefix="BB_ST")
    d["SK_ID_BUREAU"] = bb["SK_ID_BUREAU"].values
    bb_agg = bb_agg.join(d.groupby("SK_ID_BUREAU").mean(), how="left")
    b_agg = b_agg.merge(bb_agg, left_index=True, right_index=True, how="left")
    b_agg.fillna(0, inplace=True)
    del bb, bb_agg, d
    gc.collect()

    p_aggs = {
        "AMT_ANNUITY": ["mean", "min", "max"],
        "AMT_APPLICATION": ["mean", "min", "max"],
        "AMT_CREDIT": ["mean", "min", "max"],
        "AMT_DOWN_PAYMENT": ["mean", "min", "max"],
        "AMT_GOODS_PRICE": ["mean", "min", "max"],
        "HOUR_APPR_PROCESS_START": ["mean", "min", "max"],
        "RATE_DOWN_PAYMENT": ["mean", "max"],
        "RATE_INTEREST_PRIMARY": ["mean", "max"],
        "RATE_INTEREST_PRIVILEGED": ["mean", "max"],
        "DAYS_DECISION": ["mean", "min", "max"],
        "CNT_PAYMENT": ["mean", "sum"],
        "DAYS_FIRST_DRAWING": ["mean", "min"],
        "DAYS_FIRST_DUE": ["mean", "min"],
        "DAYS_LAST_DUE_1ST_VERSION": ["mean", "min", "max"],
        "DAYS_LAST_DUE": ["mean", "min"],
        "DAYS_TERMINATION": ["mean", "min"],
        "NFLAG_INSURED_ON_APPROVAL": ["mean"],
        "SELLERPLACE_AREA": ["mean", "max"],
        "NFLAG_LAST_APPL_IN_DAY": ["mean"],
    }
    prev = read_csv_safe(
        f"{DATA_DIR}/previous_application.csv",
        usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS"] + list(p_aggs),
    )
    gp = prev.groupby("SK_ID_CURR")
    p_agg = gp.agg(p_aggs)
    p_agg.columns = ["PRV_" + "_".join(c) for c in p_agg.columns]
    cnt = gp.size()
    p_agg["PRV_COUNT"] = cnt
    status = prev["NAME_CONTRACT_STATUS"]
    p_agg["PRV_APPROVED_RATIO"] = (
        prev[status == "Approved"].groupby("SK_ID_CURR").size() / cnt
    )
    p_agg["PRV_REFUSED_RATIO"] = (
        prev[status == "Refused"].groupby("SK_ID_CURR").size() / cnt
    )
    p_agg.fillna(0, inplace=True)
    del prev, gp
    gc.collect()

    pos_aggs = {
        "MONTHS_BALANCE": ["size", "min", "max"],
        "CNT_INSTALMENT": ["mean", "min", "max"],
        "CNT_INSTALMENT_FUTURE": ["mean", "min", "max", "sum"],
        "SK_DPD": ["mean", "max", "sum"],
        "SK_DPD_DEF": ["mean", "max", "sum"],
    }
    pos = read_csv_safe(
        f"{DATA_DIR}/POS_CASH_balance.csv",
        usecols=["SK_ID_CURR", "NAME_CONTRACT_STATUS"] + list(pos_aggs),
    )
    pos_agg = pos.groupby("SK_ID_CURR").agg(pos_aggs)
    pos_agg.columns = ["POS_" + "_".join(c) for c in pos_agg.columns]
    pos_completed = (
        pos[pos["NAME_CONTRACT_STATUS"] == "Completed"].groupby("SK_ID_CURR").size()
    )
    pos_agg["POS_COMPLETED_RATIO"] = pos_completed / pos_agg["POS_MONTHS_BALANCE_size"]
    pos_agg.fillna(0, inplace=True)
    del pos
    gc.collect()

    cc_aggs = {
        "MONTHS_BALANCE": ["size", "min"],
        "AMT_BALANCE": ["mean", "max"],
        "AMT_CREDIT_LIMIT_ACTUAL": ["mean", "max"],
        "AMT_DRAWINGS_ATM_CURRENT": ["mean", "max", "sum"],
        "AMT_DRAWINGS_CURRENT": ["mean", "max", "sum"],
        "AMT_DRAWINGS_OTHER_CURRENT": ["mean", "max", "sum"],
        "AMT_DRAWINGS_POS_CURRENT": ["mean", "max", "sum"],
        "AMT_INST_MIN_REGULARITY": ["mean", "max"],
        "AMT_PAYMENT_CURRENT": ["mean", "max", "sum"],
        "AMT_PAYMENT_TOTAL_CURRENT": ["mean", "max", "sum"],
        "AMT_RECEIVABLE_PRINCIPAL": ["mean", "max"],
        "AMT_RECIVABLE": ["mean", "max"],
        "AMT_TOTAL_RECEIVABLE": ["mean", "max"],
        "CNT_DRAWINGS_ATM_CURRENT": ["mean", "max", "sum"],
        "CNT_DRAWINGS_CURRENT": ["mean", "max", "sum"],
        "CNT_DRAWINGS_OTHER_CURRENT": ["mean", "max", "sum"],
        "CNT_DRAWINGS_POS_CURRENT": ["mean", "max", "sum"],
        "CNT_INSTALMENT_MATURE_CUM": ["mean", "max", "sum"],
        "SK_DPD": ["mean", "max", "sum"],
        "SK_DPD_DEF": ["mean", "max", "sum"],
    }
    cc = read_csv_safe(
        f"{DATA_DIR}/credit_card_balance.csv", usecols=["SK_ID_CURR"] + list(cc_aggs)
    )
    cc_agg = cc.groupby("SK_ID_CURR").agg(cc_aggs)
    cc_agg.columns = ["CC_" + "_".join(c) for c in cc_agg.columns]
    cc_agg["CC_UTILIZATION"] = cc_agg["CC_AMT_BALANCE_mean"] / (
        cc_agg["CC_AMT_CREDIT_LIMIT_ACTUAL_mean"] + 1
    )
    cc_agg["CC_PAYMENT_RATIO"] = cc_agg["CC_AMT_PAYMENT_CURRENT_sum"] / (
        cc_agg["CC_AMT_BALANCE_mean"] + 1
    )
    cc_agg.fillna(0, inplace=True)
    cc_agg.replace([np.inf, -np.inf], 0, inplace=True)
    del cc
    gc.collect()

    ins_aggs = {
        "NUM_INSTALMENT_VERSION": ["mean", "sum"],
        "NUM_INSTALMENT_NUMBER": ["mean", "min", "max"],
        "DAYS_INSTALMENT": ["mean", "min", "max"],
        "DAYS_ENTRY_PAYMENT": ["mean", "min", "max"],
        "AMT_INSTALMENT": ["mean", "min", "max", "sum"],
        "AMT_PAYMENT": ["mean", "min", "max", "sum"],
    }
    ins = read_csv_safe(
        f"{DATA_DIR}/installments_payments.csv", usecols=["SK_ID_CURR"] + list(ins_aggs)
    )
    ins["_DIFF"] = ins["AMT_INSTALMENT"] - ins["AMT_PAYMENT"]
    ins["_LATE"] = ins["DAYS_ENTRY_PAYMENT"] - ins["DAYS_INSTALMENT"]
    gi = ins.groupby("SK_ID_CURR")
    ins_agg = gi.agg(ins_aggs)
    ins_agg.columns = ["INS_" + "_".join(c) for c in ins_agg.columns]
    ins_agg["INS_PAYMENT_DIFF_MEAN"] = gi["_DIFF"].mean()
    ins_agg["INS_LATE_DAYS_MEAN"] = gi["_LATE"].mean()
    ins_agg["INS_PAYMENT_RATIO"] = gi["AMT_PAYMENT"].sum() / (
        gi["AMT_INSTALMENT"].sum() + 1
    )
    ins_agg.fillna(0, inplace=True)
    ins_agg.replace([np.inf, -np.inf], 0, inplace=True)
    del ins, gi
    gc.collect()

    return b_agg, p_agg, pos_agg, cc_agg, ins_agg
b_agg, p_agg, pos_agg, cc_agg, ins_agg = aggregate_supplementary_tables()


## 6. Merge Features & Preprocess

The aggregated history features are merged into the main train/test tables, then the pipeline adds domain features from the application data itself: external-score aggregates/interactions, affordability ratios, income-per-person ratios, employment/age ratios, and age-modulated external risk. Categorical variables are label-encoded consistently across train and test, missing values are median-imputed, near-constant columns are removed, and scaled matrices are prepared only for scale-sensitive models.


In [ ]:
for df in [b_agg, p_agg, pos_agg, cc_agg, ins_agg]:
    train = train.merge(df, left_on="SK_ID_CURR", right_index=True, how="left")
    test = test.merge(df, left_on="SK_ID_CURR", right_index=True, how="left")

del b_agg, p_agg, pos_agg, cc_agg, ins_agg
gc.collect()

for df in (train, test):
    df["DAYS_EMPLOYED"] = df["DAYS_EMPLOYED"].replace(365243, np.nan)
    ext = df[["EXT_SOURCE_1", "EXT_SOURCE_2", "EXT_SOURCE_3"]]
    df["EXT_SOURCE_MEAN"] = ext.mean(axis=1)
    df["EXT_SOURCE_STD"] = ext.std(axis=1)
    df["EXT_SOURCE_MIN"] = ext.min(axis=1)
    df["EXT_SOURCE_MAX"] = ext.max(axis=1)
    df["EXT_SOURCE_PROD"] = (
        ext["EXT_SOURCE_1"] * ext["EXT_SOURCE_2"] * ext["EXT_SOURCE_3"]
    )
    df["EXT_SOURCE_12"] = ext["EXT_SOURCE_1"] * ext["EXT_SOURCE_2"]
    df["EXT_SOURCE_23"] = ext["EXT_SOURCE_2"] * ext["EXT_SOURCE_3"]
    df["CREDIT_INCOME_RATIO"] = df["AMT_CREDIT"] / df["AMT_INCOME_TOTAL"]
    df["ANNUITY_INCOME_RATIO"] = df["AMT_ANNUITY"] / df["AMT_INCOME_TOTAL"]
    df["CREDIT_TERM"] = df["AMT_ANNUITY"] / df["AMT_CREDIT"]
    df["GOODS_CREDIT_RATIO"] = df["AMT_GOODS_PRICE"] / df["AMT_CREDIT"]
    df["INCOME_PER_PERSON"] = df["AMT_INCOME_TOTAL"] / (df["CNT_FAM_MEMBERS"] + 1)
    df["INCOME_PER_CHILD"] = df["AMT_INCOME_TOTAL"] / (df["CNT_CHILDREN"] + 1)
    df["EMPLOYED_BIRTH_RATIO"] = df["DAYS_EMPLOYED"] / df["DAYS_BIRTH"]
    df["EXT_SOURCE_MEAN_x_AGE"] = df["EXT_SOURCE_MEAN"] * (df["DAYS_BIRTH"] / 365.25)
    df.replace([np.inf, -np.inf], np.nan, inplace=True)
gc.collect()

y = train["TARGET"].copy()
train.drop(columns=["TARGET"], inplace=True)
train_ids = train["SK_ID_CURR"].copy()
test_ids = test["SK_ID_CURR"].copy()

cat_cols = train.select_dtypes(include=["object"]).columns.tolist()

for col in cat_cols:
    le = LabelEncoder()
    combined = pd.concat(
        [
            train[col].astype(str).fillna("MISSING"),
            test[col].astype(str).fillna("MISSING"),
        ]
    )
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str).fillna("MISSING"))
    test[col] = le.transform(test[col].astype(str).fillna("MISSING"))

feat_names = train.drop(columns=["SK_ID_CURR"]).columns.tolist()

imputer = SimpleImputer(strategy="median")
X_all = imputer.fit_transform(train.drop(columns=["SK_ID_CURR"])).astype("float32")
test_all = imputer.transform(test.drop(columns=["SK_ID_CURR"])).astype("float32")

del train, test
gc.collect()

selector = VarianceThreshold(threshold=0.001)
X_all = selector.fit_transform(X_all)
test_all = selector.transform(test_all)
kept_features = np.array(feat_names)[selector.get_support()].tolist()
X = pd.DataFrame(X_all, columns=kept_features)
X_test = pd.DataFrame(test_all, columns=kept_features)

del X_all, test_all
gc.collect()

scaler = RobustScaler()
X_scaled = scaler.fit_transform(X).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

cv = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=RANDOM_STATE)


## 7. Cross-Validation Helper

All models use the same 5-fold stratified validation routine. The helper returns out-of-fold predictions for honest train-set AUC estimation, averages fold predictions for the test set, and keeps fold-level scores for stability diagnostics.


In [ ]:
def run_cv(
    name,
    model_class,
    model_params,
    X_data,
    use_scale=False,
    is_lgb=False,
    is_xgb=False,
    is_catboost=False,
):
    """Run 5-fold cross-validation and return OOF predictions."""
    X_mat = X_data if use_scale else X_data.to_numpy(copy=False)
    X_t = X_test_scaled if use_scale else X_test.to_numpy(copy=False)
    oof = np.zeros(len(X_mat))
    test_preds = np.zeros(len(X_t))
    scores = []
    for fold, (tr_idx, va_idx) in enumerate(cv.split(X_mat, y)):
        X_tr = X_mat[tr_idx]
        X_va = X_mat[va_idx]
        y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            if is_catboost:
                m = model_class(**model_params)
                m.fit(
                    X_tr,
                    y_tr,
                    eval_set=[(X_va, y_va)],
                    verbose=False,
                    early_stopping_rounds=200,
                )
            elif is_lgb:
                m = model_class(**model_params)
                m.fit(
                    X_tr,
                    y_tr,
                    eval_set=[(X_va, y_va)],
                    eval_metric="auc",
                    callbacks=[
                        lgb.early_stopping(150, verbose=False),
                        lgb.log_evaluation(0),
                    ],
                )
            elif is_xgb:
                m = model_class(**model_params)
                m.fit(X_tr, y_tr, eval_set=[(X_va, y_va)], verbose=False)
            else:
                m = model_class(**model_params)
                m.fit(X_tr, y_tr)
            va_pred = m.predict_proba(X_va)[:, 1]
            t_pred = m.predict_proba(X_t)[:, 1]
        oof[va_idx] = va_pred
        scores.append(roc_auc_score(y_va, va_pred))
        test_preds += t_pred / N_FOLDS
        del m, X_tr, X_va
        gc.collect()

    return {
        "oof": oof,
        "test": test_preds,
        "oof_auc": roc_auc_score(y, oof),
        "scores": scores,
    }


## 8. Model 1 - Logistic Regression (Baseline)

A simple linear baseline trained on the robust-scaled matrix. Its weak OOF AUC in the latest artifacts (0.4902) shows that the risk signal is strongly nonlinear and interaction-heavy.


In [ ]:
lr_results = run_cv(
    "Logistic Regression",
    LogisticRegression,
    {
        "C": 0.1,
        "max_iter": 1000,
        "solver": "lbfgs",
        "penalty": "l2",
        "random_state": RANDOM_STATE,
    },
    X_scaled,
    use_scale=True,
)

submission_lr = sub.copy()
submission_lr["TARGET"] = lr_results["test"]
submission_lr.to_csv(SUB_DIR + "/submission_lr.csv", index=False)

del X_scaled, X_test_scaled
gc.collect()


## 9. Model 2 - XGBoost

GPU histogram boosting with 4,000 candidate trees and early stopping. This is the strongest single model in the latest artifacts, with OOF AUC 0.7876 and AP 0.2813.


In [ ]:
xgb_params = {
    "n_estimators": 4000,
    "learning_rate": 0.03,
    "max_depth": 6,
    "min_child_weight": 40,
    "subsample": 0.85,
    "colsample_bytree": 0.7,
    "gamma": 0.02,
    "reg_alpha": 0.04,
    "reg_lambda": 0.07,
    "objective": "binary:logistic",
    "eval_metric": "auc",
    "tree_method": "hist",
    "device": "cuda",
    "verbosity": 0,
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "early_stopping_rounds": 150,
}
xgb_results = run_cv(
    "XGBoost", xgb.XGBClassifier, xgb_params, X, use_scale=False, is_xgb=True
)

submission_xgb = sub.copy()
submission_xgb["TARGET"] = xgb_results["test"]
submission_xgb.to_csv(SUB_DIR + "/submission_xgb.csv", index=False)
gc.collect()


## 10. Model 3 - LightGBM

Leaf-wise gradient boosting with tuned regularization, subsampling, and early stopping. It is essentially tied with XGBoost, reaching OOF AUC 0.7867.


In [ ]:
lgb_params = {
    "n_estimators": 4000,
    "learning_rate": 0.03,
    "num_leaves": 34,
    "max_depth": 8,
    "colsample_bytree": 0.9497036,
    "subsample": 0.8715623,
    "reg_alpha": 0.041545473,
    "reg_lambda": 0.0735294,
    "min_split_gain": 0.0222415,
    "min_child_weight": 39.3259775,
    "objective": "binary",
    "metric": "auc",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": -1,
}
lgb_results = run_cv(
    "LightGBM", lgb.LGBMClassifier, lgb_params, X, use_scale=False, is_lgb=True
)

submission_lgb = sub.copy()
submission_lgb["TARGET"] = lgb_results["test"]
submission_lgb.to_csv(SUB_DIR + "/submission_lgb.csv", index=False)
gc.collect()


## 11. Model 4 - CatBoost

GPU CatBoost with ordered boosting-style handling and early stopping. It remains in the leading booster group with OOF AUC 0.7861.


In [ ]:
cb_params = {
    "iterations": 4000,
    "learning_rate": 0.03,
    "depth": 6,
    "l2_leaf_reg": 10,
    "random_strength": 1,
    "bagging_temperature": 1,
    "border_count": 128,
    "random_seed": RANDOM_STATE,
    "eval_metric": "AUC",
    "task_type": "GPU",
    "devices": "0",
    "train_dir": OUT_DIR + "/catboost_info",
    "verbose": False,
}
cb_results = run_cv(
    "CatBoost", CatBoostClassifier, cb_params, X, use_scale=False, is_catboost=True
)

submission_cb = sub.copy()
submission_cb["TARGET"] = cb_results["test"]
submission_cb.to_csv(SUB_DIR + "/submission_cb.csv", index=False)
gc.collect()


## 12. Model 5 - Random Forest

A bagging ensemble of decision trees. Unlike the three gradient-boosting models, it builds trees independently on bootstrapped samples, which makes its errors less correlated with the boosters and adds useful diversity to the blend. Class imbalance is handled with `class_weight='balanced'`. Latest artifact OOF AUC: 0.7599.


In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf_params = {
    "n_estimators": 150,
    "max_depth": 12,
    "min_samples_leaf": 30,
    "max_features": "sqrt",
    "class_weight": "balanced",
    "random_state": RANDOM_STATE,
    "n_jobs": -1,
    "verbose": 0,
}
rf_results = run_cv(
    "RandomForest", RandomForestClassifier, rf_params, X, use_scale=False
)

submission_rf = sub.copy()
submission_rf["TARGET"] = rf_results["test"]
submission_rf.to_csv(SUB_DIR + "/submission_rf.csv", index=False)
print(f"Random Forest OOF AUC: {rf_results['oof_auc']:.5f}")

gc.collect()


## 13. Model 6 - Deep Learning Neural Network (PyTorch)

A feedforward MLP with BatchNorm, Dropout, AdamW, class-balanced sampling, and positive-class weighting. It is included as a deliberately different model family, but the current tabular setup favors tree models; latest artifact OOF AUC is 0.6252.


In [ ]:
X_scaled = scaler.transform(X).astype("float32")
X_test_scaled = scaler.transform(X_test).astype("float32")

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset, WeightedRandomSampler

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")


class TabularNN(nn.Module):
    """Feedforward neural network with BatchNorm and Dropout."""
    def __init__(self, input_dim, hidden_dims=[512, 256, 128, 64], dropout=0.3):
        super().__init__()
        layers = []
        d = input_dim
        for h in hidden_dims:
            layers += [
                nn.Linear(d, h),
                nn.BatchNorm1d(h),
                nn.ReLU(),
                nn.Dropout(dropout),
            ]
            d = h
        layers.append(nn.Linear(d, 1))
        self.network = nn.Sequential(*layers)
    def forward(self, x):
        return self.network(x).squeeze(1)


def train_nn_fold(X_tr, y_tr, X_va, y_va, input_dim):
    """Train NN for a single fold with class balancing."""
    pos_weight = (len(y_tr) - y_tr.sum()) / y_tr.sum()
    counts = np.bincount(y_tr.astype(int))
    sample_w = np.where(y_tr.values == 1, 1.0 / counts[1], 1.0 / counts[0])
    sampler = WeightedRandomSampler(
        torch.DoubleTensor(sample_w), len(X_tr), replacement=True
    )
    loader = DataLoader(
        TensorDataset(torch.FloatTensor(X_tr), torch.FloatTensor(y_tr.values)),
        batch_size=1024,
        sampler=sampler,
    )
    model = TabularNN(input_dim).to(DEVICE)
    criterion = nn.BCEWithLogitsLoss(pos_weight=torch.tensor([pos_weight]).to(DEVICE))
    opt = optim.AdamW(model.parameters(), lr=0.001, weight_decay=1e-4)
    sched = optim.lr_scheduler.ReduceLROnPlateau(
        opt, mode="max", factor=0.5, patience=5
    )
    best_auc, patience = 0, 0
    best_state = None
    for epoch in range(25):
        model.train()
        for bx, by in loader:
            bx, by = bx.to(DEVICE), by.to(DEVICE)
            opt.zero_grad()
            loss = criterion(model(bx), by)
            loss.backward()
            opt.step()
        model.eval()
        with torch.no_grad():
            va_pred = (
                torch.sigmoid(model(torch.FloatTensor(X_va).to(DEVICE))).cpu().numpy()
            )
        auc = roc_auc_score(y_va, va_pred)
        sched.step(auc)
        if auc > best_auc:
            best_auc = auc
            best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
            patience = 0
        else:
            patience += 1
        if patience >= 5:
            break
    model.load_state_dict(best_state)

    return model
dl_oof = np.zeros(len(X_scaled))
dl_test = np.zeros(len(X_test_scaled))
dl_scores = []

for fold, (tr_idx, va_idx) in enumerate(cv.split(X_scaled, y)):
    t0 = time.time()
    X_tr, X_va = X_scaled[tr_idx], X_scaled[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]
    model = train_nn_fold(X_tr, y_tr, X_va, y_va, X_scaled.shape[1])
    model.eval()
    with torch.no_grad():
        va_pred = torch.sigmoid(model(torch.FloatTensor(X_va).to(DEVICE))).cpu().numpy()
        t_pred = (
            torch.sigmoid(model(torch.FloatTensor(X_test_scaled).to(DEVICE)))
            .cpu()
            .numpy()
        )
    dl_oof[va_idx] = va_pred
    dl_test += t_pred / N_FOLDS
    auc = roc_auc_score(y_va, va_pred)
    dl_scores.append(auc)
    del model
    gc.collect()

dl_oof_auc = roc_auc_score(y, dl_oof)
dl_results = {
    "oof": dl_oof,
    "test": dl_test,
    "oof_auc": dl_oof_auc,
    "scores": dl_scores,
}

submission_dl = sub.copy()
submission_dl["TARGET"] = dl_test
submission_dl.to_csv(SUB_DIR + "/submission_dl.csv", index=False)
gc.collect()


## 14. Model 7 - Weighted Ensemble

The final blend excludes Logistic Regression and combines the remaining model families with fixed weights:

- LightGBM: 0.30
- XGBoost: 0.28
- CatBoost: 0.27
- Random Forest: 0.10
- Deep Learning: 0.05

In the latest artifacts the ensemble reaches OOF AUC 0.7865. It is slightly below XGBoost on AUC but has the best Average Precision (0.2832), so it remains useful as a robust blended submission.


In [ ]:
weights = {
    "LightGBM": 0.30,
    "XGBoost": 0.28,
    "CatBoost": 0.27,
    "RandomForest": 0.10,
    "Deep Learning": 0.05,
}
ensemble_oof = (
    weights["LightGBM"] * lgb_results["oof"]
    + weights["XGBoost"] * xgb_results["oof"]
    + weights["CatBoost"] * cb_results["oof"]
    + weights["RandomForest"] * rf_results["oof"]
    + weights["Deep Learning"] * dl_results["oof"]
)
ensemble_test = (
    weights["LightGBM"] * lgb_results["test"]
    + weights["XGBoost"] * xgb_results["test"]
    + weights["CatBoost"] * cb_results["test"]
    + weights["RandomForest"] * rf_results["test"]
    + weights["Deep Learning"] * dl_results["test"]
)
ensemble_auc = roc_auc_score(y, ensemble_oof)

submission_ens = sub.copy()
submission_ens["TARGET"] = ensemble_test
submission_ens.to_csv(SUB_DIR + "/submission_ensemble.csv", index=False)
print(f"Ensemble OOF AUC: {ensemble_auc:.5f}")


## 15. Model Comparison

The latest `outputs/artifacts/model_comparison.csv` ranks the models by OOF AUC:

| Rank | Model | OOF AUC |
|---:|---|---:|
| 1 | XGBoost | 0.7876 |
| 2 | LightGBM | 0.7867 |
| 3 | Ensemble | 0.7865 |
| 4 | CatBoost | 0.7861 |
| 5 | Random Forest | 0.7599 |
| 6 | Deep Learning NN | 0.6252 |
| 7 | Logistic Regression | 0.4902 |

The boosters form a tight leading group. Random Forest trails them on AUC, but its lower correlation with the boosters makes it useful for diversity.


In [ ]:
models = {
    "Logistic Regression": lr_results["oof_auc"],
    "XGBoost": xgb_results["oof_auc"],
    "LightGBM": lgb_results["oof_auc"],
    "CatBoost": cb_results["oof_auc"],
    "Random Forest": rf_results["oof_auc"],
    "Deep Learning NN": dl_oof_auc,
    "Ensemble": ensemble_auc,
}
comparison_df = pd.DataFrame(
    {
        "Model": list(models.keys()),
        "OOF AUC": list(models.values()),
    }
).sort_values("OOF AUC", ascending=False)
comparison_df = comparison_df.reset_index(drop=True)
comparison_df.to_csv(ART_DIR + "/model_comparison.csv", index=False)

fig, ax = plt.subplots(figsize=(12, 6))
colors = [
    "#e74c3c" if m == "Ensemble" else "#9b59b6" if "Deep" in m else "#3498db"
    for m in comparison_df["Model"]
]
bars = ax.barh(comparison_df["Model"], comparison_df["OOF AUC"], color=colors)
ax.invert_yaxis()
ax.set_xlabel("ROC AUC")
ax.set_title("Model Comparison - Home Credit Default Risk")
ax.set_xlim(min(models.values()) - 0.01, max(models.values()) + 0.005)

for bar, val in zip(bars, comparison_df["OOF AUC"]):
    ax.text(
        val + 0.0005,
        bar.get_y() + bar.get_height() / 2,
        f"{val:.6f}",
        va="center",
        fontsize=11,
    )
plt.tight_layout()
plt.savefig(FIG_DIR + "/model_comparison.png", dpi=100)
plt.show()
plt.close()


## 16. Interpretability - SHAP Analysis

SHAP is computed on a sampled LightGBM model to explain the strongest tabular signals. The current feature set shifts the top SHAP drivers toward engineered external-score and domain-ratio features, especially `EXT_SOURCE_MEAN`, `EXT_SOURCE_MEAN_x_AGE`, `EMPLOYED_BIRTH_RATIO`, `GOODS_CREDIT_RATIO`, and `CODE_GENDER`.


In [ ]:
n_sample = min(3000, len(X))
X_shap_tr = X.sample(n=n_sample, random_state=RANDOM_STATE)
y_shap_tr = y.loc[X_shap_tr.index]
lgb_shap = lgb.LGBMClassifier(
    n_estimators=100,
    learning_rate=0.05,
    num_leaves=64,
    max_depth=6,
    random_state=RANDOM_STATE,
    n_jobs=-1,
    verbose=-1,
)
lgb_shap.fit(X_shap_tr, y_shap_tr)
X_explain = X.sample(n=500, random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(lgb_shap)
shap_values = explainer.shap_values(X_explain)
plt.figure(figsize=(12, 8))
shap.summary_plot(shap_values, X_explain, show=False, max_display=15)
plt.tight_layout()
plt.savefig(FIG_DIR + "/shap_summary.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()

plt.figure(figsize=(12, 7))
shap.summary_plot(shap_values, X_explain, plot_type="bar", show=False, max_display=15)
plt.tight_layout()
plt.savefig(FIG_DIR + "/shap_importance.png", dpi=150, bbox_inches="tight")
plt.show()
plt.close()

shap_imp = pd.DataFrame(
    {"feature": X.columns, "importance": np.abs(shap_values).mean(axis=0)}
).sort_values("importance", ascending=False)
shap_imp.to_csv(ART_DIR + "/shap_feature_importance.csv", index=False)
top5 = shap_imp.head(5)["feature"].tolist()

for feat in top5[:3]:
    plt.figure(figsize=(10, 6))
    shap.dependence_plot(feat, shap_values, X_explain, show=False)
    plt.tight_layout()
    plt.savefig(FIG_DIR + f'/shap_dependence_{feat[:30].replace("/","_")}.png', dpi=100)
    plt.close()

del explainer, shap_values, X_explain, X_shap_tr, lgb_shap
gc.collect()


## 17. Error Analysis

This first diagnostic block uses the LightGBM OOF predictions to inspect score distributions, threshold behavior, and feature-target correlations. The later classification-visualization section repeats the business-oriented threshold/decile analysis on the final ensemble.


In [ ]:
results_df = pd.DataFrame(
    {
        "TARGET": y.values,
        "PREDICTION": lgb_results["oof"],
    }
)
results_df["PRED_BIN"] = (results_df["PREDICTION"] >= 0.5).astype(int)
tn = ((results_df["TARGET"] == 0) & (results_df["PRED_BIN"] == 0)).sum()
fp = ((results_df["TARGET"] == 0) & (results_df["PRED_BIN"] == 1)).sum()
fn = ((results_df["TARGET"] == 1) & (results_df["PRED_BIN"] == 0)).sum()
tp = ((results_df["TARGET"] == 1) & (results_df["PRED_BIN"] == 1)).sum()

fig, ax = plt.subplots(figsize=(10, 6))

for label, color, name in [(0, "#2ecc71", "Pays"), (1, "#e74c3c", "Defaults")]:
    mask = results_df["TARGET"] == label
    ax.hist(
        results_df.loc[mask, "PREDICTION"],
        bins=50,
        alpha=0.6,
        color=color,
        label=name,
        density=True,
    )
ax.set_xlabel("Predicted Probability of Default")
ax.set_ylabel("Density")
ax.set_title("Prediction Distribution by True Class")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/error_distribution.png", dpi=100)
plt.show()
plt.close()

thresholds = np.arange(0.05, 1.0, 0.05)
tpr_list, fpr_list, f1_list = [], [], []

for t in thresholds:
    pr = (results_df["PREDICTION"] >= t).astype(int)
    tn_, fp_, fn_, tp_ = confusion_matrix(results_df["TARGET"], pr).ravel()
    tpr_list.append(tp_ / (tp_ + fn_) if (tp_ + fn_) > 0 else 0)
    fpr_list.append(fp_ / (fp_ + tn_) if (fp_ + tn_) > 0 else 0)
    f1_list.append(2 * tp_ / (2 * tp_ + fp_ + fn_) if (2 * tp_ + fp_ + fn_) > 0 else 0)
fig, ax1 = plt.subplots(figsize=(10, 6))
ax1.plot(thresholds, tpr_list, "g-", label="Recall (TPR)")
ax1.plot(thresholds, fpr_list, "r-", label="FPR")
ax1.set_xlabel("Decision Threshold")
ax1.set_ylabel("Rate")
ax1.legend(loc="center left")
ax2 = ax1.twinx()
ax2.plot(thresholds, f1_list, "b--", label="F1 Score")
ax2.set_ylabel("F1 Score", color="b")
ax2.tick_params(axis="y", labelcolor="b")
ax2.legend(loc="center right")
plt.title("Model Metrics by Decision Threshold")
plt.tight_layout()
plt.savefig(FIG_DIR + "/threshold_analysis.png", dpi=100)
plt.show()
plt.close()

correlations = []

for col in X.columns[:100]:
    c = np.corrcoef(X[col].fillna(0), y.astype(float))[0, 1]
    correlations.append({"feature": col, "correlation": c})
corr_df = pd.DataFrame(correlations).sort_values(
    "correlation", key=abs, ascending=False
)

fig, ax = plt.subplots(figsize=(12, 8))
top_corr = corr_df.head(15)
colors_bar = ["#e74c3c" if v < 0 else "#2ecc71" for v in top_corr["correlation"]]
ax.barh(range(len(top_corr)), top_corr["correlation"].values, color=colors_bar)
ax.set_yticks(range(len(top_corr)))
ax.set_yticklabels(top_corr["feature"].values)
ax.set_xlabel("Correlation with TARGET")
ax.set_title("Top 15 Feature Correlations with Default")
plt.tight_layout()
plt.savefig(FIG_DIR + "/top_correlations.png", dpi=100)
plt.show()
plt.close()

corr_df.to_csv(ART_DIR + "/feature_correlations.csv", index=False)


## 18. Generate Final Submission

The final `submission.csv` is written from the weighted ensemble predictions. Individual model submissions are also preserved under `outputs/submissions/` for comparison or alternative Kaggle uploads.


In [ ]:
final_submission = sub.copy()
final_submission["TARGET"] = ensemble_test
final_submission.to_csv(SUB_DIR + "/submission.csv", index=False)


## 19. Extended Exploratory Data Analysis

A deeper look at the raw applications, rendered inline. The raw `train`/`test` frames were freed during preprocessing to keep memory low, so this section reloads a light subset of `application_train.csv`.

We examine:
- **Class imbalance**: only about 8.1% of applicants default, which drives the validation and threshold choices.
- **Missingness** across columns.
- **EXT_SOURCE_2 / EXT_SOURCE_3** distributions by target; the engineered external-score aggregates later become the strongest model drivers.
- **Age** distribution and **default rate by age group**; younger borrowers default more.
- **Default rate by categorical attributes** (gender, education, income type, family status) versus the overall base rate.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns

sns.set_style("whitegrid")
eda_cols = [
    "TARGET",
    "EXT_SOURCE_1",
    "EXT_SOURCE_2",
    "EXT_SOURCE_3",
    "DAYS_BIRTH",
    "DAYS_EMPLOYED",
    "AMT_INCOME_TOTAL",
    "AMT_CREDIT",
    "AMT_ANNUITY",
    "CODE_GENDER",
    "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE",
    "NAME_FAMILY_STATUS",
]
eda = read_csv_safe(f"{DATA_DIR}/application_train.csv", usecols=eda_cols)
eda["AGE_YEARS"] = -eda["DAYS_BIRTH"] / 365.25
eda["DAYS_EMPLOYED"] = eda["DAYS_EMPLOYED"].replace(365243, np.nan)
base_rate = eda["TARGET"].mean()
miss_sample = read_csv_safe(f"{DATA_DIR}/application_train.csv", nrows=100_000)
miss_pct = (miss_sample.isnull().mean() * 100).sort_values(ascending=False).head(15)

del miss_sample
gc.collect()

vc = eda["TARGET"].value_counts().sort_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.bar(["Repays (0)", "Defaults (1)"], vc.values, color=["#2ecc71", "#e74c3c"])

for i, v in enumerate(vc.values):
    ax.text(i, v, f"{v:,}\n{v / len(eda) * 100:.1f}%", ha="center", va="bottom")
ax.set_title(f"Target Distribution: {base_rate * 100:.1f}% default")
ax.set_ylabel("Count")
plt.tight_layout()
plt.savefig(FIG_DIR + "/eda_target_distribution.png", dpi=120)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(9, 6))
ax.barh(miss_pct.index[::-1], miss_pct.values[::-1], color="#3498db")
ax.set(xlabel="% missing", title="Top 15 Missing-Value Columns")
plt.tight_layout()
plt.savefig(FIG_DIR + "/eda_missing_values.png", dpi=120)
plt.show()
plt.close()

for col in ["EXT_SOURCE_2", "EXT_SOURCE_3"]:

    fig, ax = plt.subplots(figsize=(8, 5))
    for label, color, name in [(0, "#2ecc71", "Repays"), (1, "#e74c3c", "Defaults")]:
        ax.hist(
            eda.loc[eda["TARGET"] == label, col].dropna(),
            bins=50,
            density=True,
            alpha=0.6,
            color=color,
            label=name,
        )
    ax.set(xlabel=col, ylabel="Density", title=f"{col} by Target")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/eda_{col.lower()}_by_target.png", dpi=120)
    plt.show()
    plt.close()

fig, ax = plt.subplots(figsize=(8, 5))

for label, color, name in [(0, "#2ecc71", "Repays"), (1, "#e74c3c", "Defaults")]:
    ax.hist(
        eda.loc[eda["TARGET"] == label, "AGE_YEARS"],
        bins=50,
        density=True,
        alpha=0.6,
        color=color,
        label=name,
    )
ax.set(xlabel="Age (years)", ylabel="Density", title="Age by Target")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/eda_age_by_target.png", dpi=120)
plt.show()
plt.close()

age_bins = pd.cut(eda["AGE_YEARS"], bins=range(20, 75, 5))
rate_by_age = eda.groupby(age_bins, observed=True)["TARGET"].mean() * 100

fig, ax = plt.subplots(figsize=(10, 5))
ax.plot([str(i) for i in rate_by_age.index], rate_by_age.values, "o-", color="#e74c3c")
ax.axhline(base_rate * 100, ls="--", color="k", label=f"Overall {base_rate * 100:.1f}%")
ax.set(ylabel="Default rate (%)", title="Default Rate by Age Group")
ax.tick_params(axis="x", rotation=45)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/eda_default_rate_by_age.png", dpi=120)
plt.show()
plt.close()

for col in [
    "CODE_GENDER",
    "NAME_EDUCATION_TYPE",
    "NAME_INCOME_TYPE",
    "NAME_FAMILY_STATUS",
]:
    grp = eda.groupby(col)["TARGET"].agg(["mean", "count"])
    grp = grp[grp["count"] >= 100].sort_values("mean")

    fig, ax = plt.subplots(figsize=(10, max(4, 0.45 * len(grp))))
    bars = ax.barh(grp.index.astype(str), grp["mean"].values * 100, color="#9b59b6")
    ax.axvline(
        base_rate * 100, ls="--", color="k", label=f"Overall {base_rate * 100:.1f}%"
    )
    for b, n in zip(bars, grp["count"].values):
        ax.text(
            b.get_width(),
            b.get_y() + b.get_height() / 2,
            f"  n={n:,}",
            va="center",
            fontsize=8,
        )
    ax.set(xlabel="Default rate (%)", title=f"Default Rate by {col}")
    ax.legend()
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/eda_default_rate_{col.lower()}.png", dpi=120)
    plt.show()
    plt.close()

del eda
gc.collect()

print("Saved extended EDA figures")


## 20. Result Analysis

Visual diagnostics of the trained models (all rendered inline):
- **ROC curves** for every model overlaid using out-of-fold predictions.
- **Precision-Recall** and **calibration** curves for the final ensemble, which are important under heavy class imbalance.
- **OOF score distribution** by true class to inspect class separation.
- **CV fold stability** for each model.
- **Model-prediction correlation**, where Random Forest is less correlated with the boosters and therefore contributes diversity.
- **Confusion matrix** for the ensemble at the default 0.5 threshold.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import (
    roc_curve,
    auc,
    precision_recall_curve,
    average_precision_score,
    confusion_matrix,
)
from sklearn.calibration import calibration_curve

sns.set_style("whitegrid")

oof_map = {
    "LogReg": lr_results["oof"],
    "XGBoost": xgb_results["oof"],
    "LightGBM": lgb_results["oof"],
    "CatBoost": cb_results["oof"],
    "RandForest": rf_results["oof"],
    "Deep NN": dl_results["oof"],
    "Ensemble": ensemble_oof,
}
pos_rate = y.mean()

fig, ax = plt.subplots(figsize=(8, 7))

for name, p in oof_map.items():
    fpr, tpr, _ = roc_curve(y, p)
    lw = 3 if name == "Ensemble" else 1.6
    ax.plot(fpr, tpr, lw=lw, label=f"{name} (AUC = {auc(fpr, tpr):.4f})")
ax.plot([0, 1], [0, 1], "k--", lw=1, label="Random")
ax.set(
    xlabel="False Positive Rate",
    ylabel="True Positive Rate",
    title="Out-of-Fold ROC Curves by Model",
)
ax.legend(loc="lower right")
plt.tight_layout()
plt.savefig(FIG_DIR + "/roc_curves.png", dpi=120)
plt.show()
plt.close()

prec, rec, _ = precision_recall_curve(y, ensemble_oof)
ap = average_precision_score(y, ensemble_oof)

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(rec, prec, color="#9b59b6", lw=2, label=f"AP = {ap:.4f}")
ax.axhline(pos_rate, ls="--", color="k", label=f"Baseline ({pos_rate:.3f})")
ax.set(xlabel="Recall", ylabel="Precision", title="Ensemble Precision-Recall Curve")
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/ensemble_precision_recall_curve.png", dpi=120)
plt.show()
plt.close()

frac_pos, mean_pred = calibration_curve(y, ensemble_oof, n_bins=10, strategy="quantile")
fig, ax = plt.subplots(figsize=(7, 6))
ax.plot(mean_pred, frac_pos, "o-", color="#e67e22", label="Ensemble")
ax.plot([0, 1], [0, 1], "k--", label="Perfectly calibrated")
ax.set(
    xlabel="Mean predicted probability",
    ylabel="Observed default rate",
    title="Calibration Curve",
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/ensemble_calibration_curve.png", dpi=120)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(8, 5))

for label, color, name in [(0, "#2ecc71", "Repays"), (1, "#e74c3c", "Defaults")]:
    ax.hist(
        ensemble_oof[y.values == label],
        bins=50,
        density=True,
        alpha=0.6,
        color=color,
        label=name,
    )
ax.set(
    xlabel="Predicted P(default)",
    ylabel="Density",
    title="Ensemble OOF Score Distribution",
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/ensemble_score_distribution.png", dpi=120)
plt.show()
plt.close()

scores_map = {
    "LogReg": lr_results["scores"],
    "XGBoost": xgb_results["scores"],
    "LightGBM": lgb_results["scores"],
    "CatBoost": cb_results["scores"],
    "RandForest": rf_results["scores"],
    "Deep NN": dl_results["scores"],
}

fig, ax = plt.subplots(figsize=(9, 5))

for i, (name, s) in enumerate(scores_map.items()):
    s = np.array(s)
    ax.scatter([i] * len(s), s, color="#34495e", alpha=0.6, zorder=3)
    ax.errorbar(
        i, s.mean(), yerr=s.std(), fmt="o", color="#e74c3c", capsize=5, ms=9, zorder=4
    )
ax.set_xticks(range(len(scores_map)))
ax.set_xticklabels(list(scores_map.keys()), rotation=20)
ax.set(ylabel="Fold AUC", title="CV Fold AUC Stability")
plt.tight_layout()
plt.savefig(FIG_DIR + "/cv_fold_auc_stability.png", dpi=120)
plt.show()
plt.close()

oof_df = pd.DataFrame({k: v for k, v in oof_map.items() if k != "Ensemble"})

fig, ax = plt.subplots(figsize=(8, 6))
sns.heatmap(oof_df.corr(), annot=True, fmt=".3f", cmap="viridis", vmin=0, vmax=1, ax=ax)
ax.set_title("Correlation of Model OOF Predictions")
plt.tight_layout()
plt.savefig(FIG_DIR + "/model_oof_correlation.png", dpi=120)
plt.show()
plt.close()

cm = confusion_matrix(y, (ensemble_oof >= 0.5).astype(int))
cm_norm = cm / cm.sum(axis=1, keepdims=True)

fig, ax = plt.subplots(figsize=(6, 5))
sns.heatmap(
    cm_norm,
    annot=cm,
    fmt=",d",
    cmap="Blues",
    cbar=False,
    xticklabels=["Pred Repay", "Pred Default"],
    yticklabels=["True Repay", "True Default"],
    ax=ax,
)
ax.set_title("Ensemble Confusion Matrix @ threshold 0.5")
plt.tight_layout()
plt.savefig(FIG_DIR + "/ensemble_confusion_matrix.png", dpi=120)
plt.show()
plt.close()

pd.DataFrame({**oof_map, "TARGET": y.values}).to_csv(
    ART_DIR + "/oof_predictions.csv", index=False
)
print("Saved result analysis figures")


## 21. More Classification Visualizations

Business-oriented diagnostics beyond raw AUC:
- **Precision-Recall curves** for all models overlaid.
- **Cumulative gains**: what fraction of all defaulters are captured by reviewing the top-X% riskiest applicants.
- **Lift curve**: how many times better than random the model is at each targeting depth.
- **Kolmogorov-Smirnov (KS)**: latest ensemble KS is 0.432 at about 31% of the population.
- **Default rate by risk decile**: the latest ensemble rises from about 1.0% in the lowest-risk decile to 30.3% in the highest-risk decile.
- **Threshold tuning**: latest best-F1 threshold is 0.24, with F1 0.340, precision 0.272, and recall 0.453.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import precision_recall_curve, average_precision_score

sns.set_style("whitegrid")
yv = y.values
pos_rate = yv.mean()

oof_all = {
    "LogReg": lr_results["oof"],
    "XGBoost": xgb_results["oof"],
    "LightGBM": lgb_results["oof"],
    "CatBoost": cb_results["oof"],
    "RandForest": rf_results["oof"],
    "Deep NN": dl_results["oof"],
    "Ensemble": ensemble_oof,
}

order = np.argsort(-ensemble_oof)
y_sorted = yv[order]
frac_pop = np.arange(1, len(y_sorted) + 1) / len(y_sorted)
cum_pos = np.cumsum(y_sorted) / y_sorted.sum()
cum_neg = np.cumsum(1 - y_sorted) / (1 - y_sorted).sum()
ks_gap = cum_pos - cum_neg
ks_stat = np.max(ks_gap)
ks_at = frac_pop[np.argmax(ks_gap)]

fig, ax = plt.subplots(figsize=(8, 6))

for name, p in oof_all.items():

    prec, rec, _ = precision_recall_curve(yv, p)
    lw = 3 if name == "Ensemble" else 1.4
    ax.plot(rec, prec, lw=lw, label=f"{name} (AP={average_precision_score(yv, p):.3f})")
ax.axhline(pos_rate, ls="--", color="k", label=f"Baseline ({pos_rate:.3f})")
ax.set(xlabel="Recall", ylabel="Precision", title="Precision-Recall Curves")
ax.legend(fontsize=8)
plt.tight_layout()
plt.savefig(FIG_DIR + "/precision_recall_curves_all_models.png", dpi=120)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frac_pop, cum_pos, color="#2980b9", lw=2, label="Ensemble")
ax.plot([0, 1], [0, 1], "k--", label="Random")
ax.plot([0, pos_rate, 1], [0, 1, 1], color="#27ae60", ls=":", label="Perfect")
ax.set(
    xlabel="Fraction of applicants (highest risk first)",
    ylabel="Fraction of defaulters captured",
    title="Cumulative Gains Curve",
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/cumulative_gains_curve.png", dpi=120)
plt.show()
plt.close()

lift = cum_pos / frac_pop

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frac_pop, lift, color="#8e44ad", lw=2)
ax.axhline(1, ls="--", color="k", label="No model (lift=1)")
ax.set(
    xlabel="Fraction of applicants targeted",
    ylabel="Lift",
    title="Lift Curve",
    ylim=(0, lift[10:].max() * 1.1),
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/lift_curve.png", dpi=120)
plt.show()
plt.close()

fig, ax = plt.subplots(figsize=(8, 6))
ax.plot(frac_pop, cum_pos, color="#e74c3c", label="Cumulative % defaulters")
ax.plot(frac_pop, cum_neg, color="#2ecc71", label="Cumulative % non-defaulters")
ax.vlines(
    ks_at,
    cum_neg[np.argmax(ks_gap)],
    cum_pos[np.argmax(ks_gap)],
    color="k",
    lw=2,
    label=f"KS = {ks_stat:.3f}",
)
ax.set(
    xlabel="Fraction of population (sorted by score)",
    ylabel="Cumulative proportion",
    title="Kolmogorov-Smirnov Separation",
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/ks_curve.png", dpi=120)
plt.show()
plt.close()

dec = pd.qcut(ensemble_oof, 10, labels=False, duplicates="drop")
dec_rate = pd.DataFrame({"d": dec, "y": yv}).groupby("d")["y"].mean() * 100

fig, ax = plt.subplots(figsize=(8, 5))
bars = ax.bar(dec_rate.index + 1, dec_rate.values, color="#34495e")
ax.axhline(
    pos_rate * 100, ls="--", color="#e74c3c", label=f"Overall {pos_rate * 100:.1f}%"
)

for b, v in zip(bars, dec_rate.values):
    ax.text(
        b.get_x() + b.get_width() / 2,
        v,
        f"{v:.1f}",
        ha="center",
        va="bottom",
        fontsize=8,
    )
ax.set(
    xlabel="Risk decile (1=lowest score, 10=highest)",
    ylabel="Actual default rate (%)",
    title="Default Rate by Predicted-Risk Decile",
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/default_rate_by_risk_decile.png", dpi=120)
plt.show()
plt.close()

ths = np.linspace(0.05, 0.95, 91)
prec_t, rec_t, f1_t = [], [], []

for t in ths:
    pred = (ensemble_oof >= t).astype(int)
    tp = ((pred == 1) & (yv == 1)).sum()
    fp = ((pred == 1) & (yv == 0)).sum()
    fn = ((pred == 0) & (yv == 1)).sum()
    prec_t.append(tp / (tp + fp) if tp + fp else 0)
    rec_t.append(tp / (tp + fn) if tp + fn else 0)
    f1_t.append(2 * tp / (2 * tp + fp + fn) if (2 * tp + fp + fn) else 0)
best = int(np.argmax(f1_t))

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(ths, prec_t, label="Precision", color="#3498db")
ax.plot(ths, rec_t, label="Recall", color="#e67e22")
ax.plot(ths, f1_t, label="F1", color="#9b59b6", lw=2)
ax.axvline(
    ths[best], ls="--", color="k", label=f"Best F1 @ {ths[best]:.2f} ({f1_t[best]:.3f})"
)
ax.set(
    xlabel="Decision threshold",
    ylabel="Score",
    title="Precision / Recall / F1 vs Threshold",
)
ax.legend()
plt.tight_layout()
plt.savefig(FIG_DIR + "/threshold_precision_recall_f1.png", dpi=120)
plt.show()
plt.close()

print(
    f"KS = {ks_stat:.4f} at {ks_at * 100:.0f}% of population | best-F1 threshold = {ths[best]:.2f}"
)
print("Saved classification figures")


## 22. Extended Explainability

Going beyond the single SHAP summary above:
- **Cross-model feature importance** compares LightGBM, XGBoost, CatBoost, and Random Forest after normalizing each model's importances.
- **Permutation importance** measures the AUC drop when a feature is shuffled, making it model-agnostic.
- **SHAP beeswarm** shows per-instance feature effects on a sampled LightGBM model.
- **Local explanations** decompose one high-risk and one low-risk applicant into signed feature contributions.

The current top SHAP artifact emphasizes engineered external-score aggregates and ratios rather than only the raw `EXT_SOURCE_*` columns.


In [ ]:
%matplotlib inline

import matplotlib.pyplot as plt
import shap
from sklearn.inspection import permutation_importance
from sklearn.ensemble import RandomForestClassifier

samp = X.sample(n=min(80_000, len(X)), random_state=RANDOM_STATE).index
Xs, ys = X.loc[samp], y.loc[samp]

imp = {}
m_lgb = lgb.LGBMClassifier(
    n_estimators=150,
    num_leaves=64,
    learning_rate=0.05,
    n_jobs=-1,
    verbose=-1,
    random_state=RANDOM_STATE,
).fit(Xs, ys)
imp["LightGBM"] = m_lgb.feature_importances_

m_xgb = xgb.XGBClassifier(
    n_estimators=150,
    max_depth=6,
    learning_rate=0.05,
    n_jobs=-1,
    verbosity=0,
    random_state=RANDOM_STATE,
).fit(Xs, ys)
imp["XGBoost"] = m_xgb.feature_importances_

m_cb = CatBoostClassifier(
    iterations=150, depth=6, learning_rate=0.05, verbose=False, random_seed=RANDOM_STATE
).fit(Xs, ys)
imp["CatBoost"] = m_cb.feature_importances_

m_rf = RandomForestClassifier(
    n_estimators=150,
    max_depth=14,
    min_samples_leaf=30,
    max_features="sqrt",
    n_jobs=-1,
    random_state=RANDOM_STATE,
    verbose=0,
).fit(Xs, ys)
imp["RandomForest"] = m_rf.feature_importances_
imp_df = pd.DataFrame(imp, index=X.columns)
imp_norm = imp_df / imp_df.sum(axis=0)
top_feats = imp_norm.mean(axis=1).sort_values(ascending=False).head(15).index[::-1]

fig, ax = plt.subplots(figsize=(10, 8))
imp_norm.loc[top_feats].plot.barh(
    ax=ax, width=0.8, color=["#2ecc71", "#3498db", "#e67e22", "#9b59b6"]
)
ax.set(
    xlabel="Normalised importance (share of total)",
    title="Top-15 Feature Importance across Tree Models",
)
ax.legend(title="Model")
plt.tight_layout()
plt.savefig(FIG_DIR + "/cross_model_feature_importance.png", dpi=120)
plt.show()
plt.close()

pi_idx = X.sample(n=min(20_000, len(X)), random_state=7).index
perm = permutation_importance(
    m_lgb,
    X.loc[pi_idx],
    y.loc[pi_idx],
    scoring="roc_auc",
    n_repeats=5,
    n_jobs=-1,
    random_state=RANDOM_STATE,
)
pi = (
    pd.Series(perm.importances_mean, index=X.columns)
    .sort_values(ascending=False)
    .head(15)
)

fig, ax = plt.subplots(figsize=(9, 7))
ax.barh(
    pi.index[::-1],
    pi.values[::-1],
    xerr=pd.Series(perm.importances_std, index=X.columns)[pi.index][::-1],
    color="#16a085",
)
ax.set(xlabel="Mean AUC drop when shuffled", title="Permutation Importance")
plt.tight_layout()
plt.savefig(FIG_DIR + "/permutation_importance.png", dpi=120)
plt.show()
plt.close()

X_shap = Xs.sample(n=1500, random_state=RANDOM_STATE)
explainer = shap.TreeExplainer(m_lgb)
sv = explainer.shap_values(X_shap)
sv_pos = sv[1] if isinstance(sv, list) else sv
base_val = explainer.expected_value
base_val = base_val[1] if isinstance(base_val, (list, np.ndarray)) else base_val

fig = plt.figure(figsize=(10, 8))
shap.summary_plot(sv_pos, X_shap, show=False, max_display=15)
plt.title("SHAP Beeswarm: feature impact on default risk", fontsize=13)
plt.tight_layout()
plt.savefig(FIG_DIR + "/shap_beeswarm_extra.png", dpi=120, bbox_inches="tight")
plt.show()
plt.close()

probs = m_lgb.predict_proba(X_shap)[:, 1]

for idx, label in [
    (int(np.argmax(probs)), "high_risk"),
    (int(np.argmin(probs)), "low_risk"),
]:
    contrib = pd.Series(sv_pos[idx], index=X_shap.columns)
    top = contrib.reindex(contrib.abs().sort_values(ascending=False).head(12).index)[
        ::-1
    ]

    fig, ax = plt.subplots(figsize=(10, 6))
    ax.barh(
        top.index,
        top.values,
        color=["#e74c3c" if v > 0 else "#2ecc71" for v in top.values],
    )
    ax.axvline(0, color="k", lw=0.8)
    ax.set(
        xlabel="SHAP value (pushes toward default)",
        title=f'Local Explanation: {label.replace("_", " ").title()} applicant (P(default)={probs[idx]:.2f})',
    )
    plt.tight_layout()
    plt.savefig(FIG_DIR + f"/shap_local_{label}.png", dpi=120)
    plt.show()
    plt.close()

del explainer, sv, sv_pos, m_lgb, m_xgb, m_cb, m_rf, Xs, ys, X_shap
gc.collect()

print("Saved explainability figures")


## Summary

### Competition Details
- **Problem:** Home Credit Default Risk - binary classification
- **Metric:** ROC AUC
- **Examples:** 307,511 training | 48,744 test
- **Data Type:** Tabular relational data (main application table plus historical supplementary tables)

### Approaches Compared
1. **Logistic Regression** - simple linear baseline
2. **XGBoost** - strongest single model in the latest artifacts (AUC 0.7876)
3. **LightGBM** - near-tied booster (AUC 0.7867)
4. **CatBoost** - third booster family (AUC 0.7861)
5. **Random Forest** - bagging model for ensemble diversity (AUC 0.7599)
6. **Deep Learning NN** - MLP with BatchNorm, Dropout, and class-balanced sampling (AUC 0.6252)
7. **Weighted Ensemble** - LightGBM 30%, XGBoost 28%, CatBoost 27%, Random Forest 10%, Deep NN 5% (AUC 0.7865; AP 0.2832)

### Key Techniques
- **Feature Engineering:** relational aggregations plus external-score interactions, affordability ratios, and payment-history ratios
- **Class Imbalance Handling:** booster weighting, `class_weight='balanced'`, weighted sampling, and threshold diagnostics
- **Missing Value Strategy:** median imputation after consistent train/test encoding
- **Memory Management:** numeric downcasting, `float32` matrices, and freeing large frames between stages
- **Validation:** 5-fold stratified cross-validation with OOF predictions
- **Interpretability:** SHAP, cross-model importance, permutation importance, and local explanations
- **Error Analysis:** ROC/PR curves, calibration, confusion matrix, threshold tuning, KS, lift, and risk deciles

### Files Generated
- `outputs/submissions/submission.csv` -> final ensemble submission
- `outputs/submissions/submission_lgb.csv`, `submission_xgb.csv`, `submission_cb.csv`, `submission_rf.csv`, `submission_dl.csv`, `submission_lr.csv` -> individual model submissions
- `outputs/artifacts/model_comparison.csv` -> current OOF AUC ranking
- `outputs/artifacts/oof_predictions.csv` -> OOF scores for diagnostics
- `outputs/figures/*.png` -> EDA, model comparison, SHAP, threshold, decile, ROC/PR, and calibration figures
